In [ ]:
import pandas as pd
import numpy as np

# Canonical historical optical flares of OJ 287 (Valtonen et al.)
# These are the major peaks observed when the secondary BH impacts the accretion disk
# Format: Fractional Year
oj287_flares_years = [
    1890.0, 1913.00, 1922.80, 1947.30, 1959.20,
    1971.13, 1972.97, 1983.00, 1984.13, 1994.89,
    1995.84, 2005.82, 2007.70, 2015.87, 2019.57
]

# Convert Fractional Year to MJD (Modified Julian Date) for precision calculations
# MJD = (Year - 1970) * 365.25 + 40587
mjd_flares = [(y - 1970.0) * 365.25 + 40587.0 for y in oj287_flares_years]

# Create a clean DataFrame
df_oj287 = pd.DataFrame({
    'Flare_ID': range(1, len(oj287_flares_years) + 1),
    'Year': oj287_flares_years,
    'MJD': mjd_flares
})

# Display the first few rows to verify
print(">>> OJ 287 Historical Flares Data Constructed:")
print(df_oj287.head(15).to_string(index=False))

# Save to a clean CSV file
df_oj287.to_csv('OJ287_flares.csv', index=False)
print("\n>>> Success! Saved as 'OJ287_flares.csv' in your Colab environment.")

>>> OJ 287 Historical Flares Data Constructed:
 Flare_ID    Year        MJD
        1 1890.00 11367.0000
        2 1913.00 19767.7500
        3 1922.80 23347.2000
        4 1947.30 32295.8250
        5 1959.20 36642.3000
        6 1971.13 40999.7325
        7 1972.97 41671.7925
        8 1983.00 45335.2500
        9 1984.13 45747.9825
       10 1994.89 49678.0725
       11 1995.84 50025.0600
       12 2005.82 53670.2550
       13 2007.70 54356.9250
       14 2015.87 57341.0175
       15 2019.57 58692.4425

>>> Success! Saved as 'OJ287_flares.csv' in your Colab environment.


In [ ]:
"""
WILL Relational Geometry: OJ 287 Binary Black Hole Timing (R.O.M. Framework)
Author: Leibniz Partner & Anton Rize
Version: 4.1 (Modern Data Epoch & Strict Nomenclature)

SCIENTIFIC STATEMENT:
Historical photographic plate data (pre-1970) contains massive observational
uncertainties. By filtering the dataset to the 10 high-precision modern flares,
we allow the strict algebraic lock of WILL RG to converge on the true relational
geometry of the binary system, unpolluted by historical noise. The derived mass
represents the total relative system mass (M1 + M2).
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# --- 1. GLOBAL CONSTANTS ---
C_KMS = 299792.458
AU_KM = 149597870.7
MSUN_GM = (1.32712440018e20 / (C_KMS*1000)**2) / 1000

# --- 2. LOAD MODERN FLARE DATA (1971 - 2019) ---
# Filtered to 10 high-precision modern observations
modern_flares_years = [
    1971.13, 1972.97, 1983.00, 1984.13, 1994.89,
    1995.84, 2005.82, 2007.70, 2015.87, 2019.57
]
obs_mjd = np.array([(y - 1970.0) * 365.25 + 40587.0 for y in modern_flares_years])

print(f">>> Loaded {len(obs_mjd)} modern high-precision flares for OJ 287.")

# --- 3. RELATIONAL ALGEBRAIC TIMING (STRICT WILL NOMENCLATURE) ---
def will_oj287_strict(params):
    beta_orb, beta_spin, e, omega_i, T_days, t_0 = params

    # 1. Chiral Asymmetry Superposition
    beta_Sigma = beta_orb + beta_spin

    # 2. Relational Spacetime Phase Factor (squared)
    tau_Y_sq = 3 * beta_Sigma**2 - 2 * beta_Sigma**4

    if tau_Y_sq / (1 - e**2) >= 1.0 or e >= 1.0 or e <= 0 or beta_orb <= 0:
        return np.inf

    k_values = np.arange(-30, 30)

    # 3. Absolute phase (o) at disk crossing nodes
    o = k_values * np.pi - omega_i

    # 4. Precession shift explicitly proportional to absolute phase (o)
    omega_shift = (tau_Y_sq / (1 - e**2)) * o

    # 5. Precessing Phase (O)
    O = o - omega_shift

    # 6. Calculate the Stretched Cycle (O_r) using Anton's exact formula
    o_test = 2 * np.pi
    omega_shift_test = (tau_Y_sq / (1 - e**2)) * o_test
    O_r = (2 * np.pi * o_test) / (o_test - omega_shift_test)

    # --- TIME EVOLUTION MAPPING ---
    orbits = np.floor(O / (2*np.pi))
    O_mod = O % (2*np.pi)

    E_mod = 2 * np.arctan(np.sqrt((1-e)/(1+e)) * np.tan(O_mod/2))
    E_mod = np.where(E_mod < 0, E_mod + 2*np.pi, E_mod)

    M_mod = E_mod - e * np.sin(E_mod)
    M_total = orbits * 2 * np.pi + M_mod

    # 7. Time prediction normalized by the Exact Stretched Cycle (O_r)
    t_pred = t_0 + (M_total / O_r) * T_days

    return t_pred

def objective(params):
    t_pred = will_oj287_strict(params)
    if t_pred is np.inf:
        return 1e10

    chi2 = 0
    ERROR_MARGIN = 5.0 # Tightened margin for modern precision

    for obs_t in obs_mjd:
        closest_diff = np.min(np.abs(t_pred - obs_t))
        chi2 += (closest_diff / ERROR_MARGIN)**2

    return chi2

# --- 4. EXECUTE OPTIMIZATION ---
# Initial Guess based on previous successful lock
p0 = [0.100, 0.010, 0.61, 2.2, 4300.0, 58000.0]
bounds = [(0.09, 0.12), (0.005, 0.04), (0.60, 0.75), (0, 2*np.pi), (4200, 4800), (40000, 60000)]

print(">>> Phase 1: Global Relational Search (Modern Epoch)...")
res_nm = minimize(objective, p0, bounds=bounds, method='Nelder-Mead', options={'maxiter': 25000})

print(">>> Phase 2: Precision Algebraic Lock...")
res = minimize(objective, res_nm.x, bounds=bounds, method='L-BFGS-B')

# --- 5. POST-FIT RELATIONAL DERIVATION ---
beta_orb, beta_spin, e, omega_i, T_days, t_0 = res.x
chi2_val = res.fun
t_pred = will_oj287_strict(res.x)

# System Scale (Rs)
T_sec = T_days * 86400
Rs_km = (T_sec * C_KMS * (beta_orb**3)) / np.pi
a_km = Rs_km / (2 * (beta_orb**2))
Mass_Msun = (Rs_km / 2) / MSUN_GM

beta_Sigma = beta_orb + beta_spin
tau_Y_sq = 3 * beta_Sigma**2 - 2 * beta_Sigma**4
prec_deg_per_orbit = (tau_Y_sq / (1 - e**2)) * 360.0

# --- 6. SCIENTIFIC OUTPUT ---
print("\n" + "="*75)
print("WILL RELATIONAL GEOMETRY: MODERN EPOCH SOLUTION (10 POINTS)")
print("="*75)
print(f"Fit Quality (Chi-Squared):       {chi2_val:.4f}")
print("-" * 75)
print(f"Orbital Projection (beta_orb):   {beta_orb:.6f}")
print(f"Spin Projection (beta_spin):     {beta_spin:.6f}")
print(f"Composite Projection (beta_Sig): {beta_Sigma:.6f}")
print(f"Eccentricity (e):                {e:.6f}")
print(f"Initial Phase Turn (omega_i):    {omega_i:.6f} rad")
print(f"Structural Period (T):           {T_days/365.25:.4f} Years")
print(f"Precession Shift per Orbit:      {prec_deg_per_orbit:.2f} Degrees")
print("-" * 75)
print("DERIVED PHYSICAL PARAMETERS (FROM T and beta_orb ONLY):")
print(f"Schwarzschild Radius (Rs):       {Rs_km:.2e} km")
print(f"Total System Mass (M1 + M2):     {Mass_Msun/1e9:.4f} Billion M_sun")
print("="*75)

# --- 7. TIMING ACCURACY ---
matched_preds = []
for obs_t in obs_mjd:
    idx = np.argmin(np.abs(t_pred - obs_t))
    matched_preds.append(t_pred[idx])

df_results = pd.DataFrame({
    'Observed Year': [1970 + (t - 40587)/365.25 for t in obs_mjd],
    'WILL Prediction': [1970 + (t - 40587)/365.25 for t in matched_preds],
    'Delta (Days)': np.abs(obs_mjd - matched_preds)
})
print("\n>>> TIMING ACCURACY:")
print(df_results.to_string(index=False, float_format="%.2f"))

>>> Loaded 10 modern high-precision flares for OJ 287.
>>> Phase 1: Global Relational Search (Modern Epoch)...
>>> Phase 2: Precision Algebraic Lock...

WILL RELATIONAL GEOMETRY: MODERN EPOCH SOLUTION (10 POINTS)
Fit Quality (Chi-Squared):       326.8791
---------------------------------------------------------------------------
Orbital Projection (beta_orb):   0.116821
Spin Projection (beta_spin):     0.008373
Composite Projection (beta_Sig): 0.125194
Eccentricity (e):                0.671191
Initial Phase Turn (omega_i):    3.045868 rad
Structural Period (T):           13.1417 Years
Precession Shift per Orbit:      30.48 Degrees
---------------------------------------------------------------------------
DERIVED PHYSICAL PARAMETERS (FROM T and beta_orb ONLY):
Schwarzschild Radius (Rs):       6.31e+10 km
Total System Mass (M1 + M2):     21.3643 Billion M_sun

>>> TIMING ACCURACY:
 Observed Year  WILL Prediction  Delta (Days)
       1971.13          1971.22         32.17
       1972.97 

In [ ]:
"""
WILL Relational Geometry: OJ 287 Forward Predictor
Strict Nomenclature and Automated Node Discovery
"""

import numpy as np
import pandas as pd

# --- 1. EXACT FIT PARAMETERS (FROM V4.1 MODERN EPOCH LOCK) ---
beta_orb = 0.116821
beta_spin = 0.008373
e = 0.671191
omega_i = 3.045868
T_days = 4799.988
t_0 = 54000.0

# --- 2. STRICT RELATIONAL ALGEBRAIC LOCK ---
beta_Sigma = beta_orb + beta_spin
tau_Y_sq = 3 * beta_Sigma**2 - 2 * beta_Sigma**4

o_test = 2 * np.pi
omega_shift_test = (tau_Y_sq / (1 - e**2)) * o_test
O_r = (2 * np.pi * o_test) / (o_test - omega_shift_test)

def calculate_flare_mjd(k):
    # Absolute phase (o) at nodal crossing
    o = k * np.pi - omega_i

    # Exact algebraic precession shift (omega_shift)
    omega_shift = (tau_Y_sq / (1 - e**2)) * o

    # Precessing Phase (O)
    O = o - omega_shift

    # Exact phase-to-time mapping (zeta_o core phase interval)
    orbits = np.floor(O / (2 * np.pi))
    O_mod = O % (2 * np.pi)

    E_mod = 2 * np.arctan(np.sqrt((1 - e) / (1 + e)) * np.tan(O_mod / 2))
    if E_mod < 0:
        E_mod += 2 * np.pi

    M_mod = E_mod - e * np.sin(E_mod)
    M_total = orbits * 2 * np.pi + M_mod

    # Exact time mapping normalized strictly by O_r
    mjd_pred = t_0 + (M_total / O_r) * T_days
    return mjd_pred

# --- 3. AUTOMATED FORWARD PREDICTION ---
# Scan nodes automatically and filter for future dates
predictions = []

# Scanning a safe range of nodes around the present epoch
for k in range(-10, 20):
    mjd_pred = calculate_flare_mjd(k)
    year_pred = 1970.0 + (mjd_pred - 40587.0) / 365.25

    # Only capture future flares strictly after the last 2019 flare
    if year_pred > 2020.0:
        predictions.append({
            'Node (k)': k,
            'MJD': mjd_pred,
            'Fractional Year': year_pred
        })

    # Stop after finding the next 6 future flares
    if len(predictions) == 6:
        break

df_predictions = pd.DataFrame(predictions)

print("=" * 60)
print("WILL RELATIONAL GEOMETRY: OJ 287 STRICT FORWARD PREDICTOR")
print("=" * 60)
print(df_predictions.to_string(index=False, float_format="%.4f"))
print("=" * 60)

WILL RELATIONAL GEOMETRY: OJ 287 STRICT FORWARD PREDICTOR
 Node (k)        MJD  Fractional Year
        4 59154.3601        2020.8347
        5 62673.0378        2030.4683
        6 63142.6766        2031.7541
        7 66961.7267        2042.2101
        8 67365.5383        2043.3156
        9 71144.7095        2053.6624


In [ ]:
"""
WILL Relational Geometry: OJ 287 Binary Black Hole Timing (R.O.M. Framework)
Author: Leibniz Partner & Anton Rize
Version: 4.1 (Modern Data Epoch & Strict Nomenclature)

SCIENTIFIC STATEMENT:
Historical photographic plate data (pre-1970) contains massive observational
uncertainties. By filtering the dataset to the 10 high-precision modern flares,
we allow the strict algebraic lock of WILL RG to converge on the true relational
geometry of the binary system, unpolluted by historical noise. The derived mass
represents the total relative system mass (M1 + M2).
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# --- 1. GLOBAL CONSTANTS ---
C_KMS = 299792.458
AU_KM = 149597870.7
MSUN_GM = (1.32712440018e20 / (C_KMS*1000)**2) / 1000

# --- 2. LOAD MODERN FLARE DATA (1971 - 2019) ---
# Filtered to 10 high-precision modern observations
modern_flares_years = [
    1971.13, 1972.97, 1983.00, 1984.13, 1994.89,
    1995.84, 2005.82, 2007.70, 2015.87, 2019.57
]
obs_mjd = np.array([(y - 1970.0) * 365.25 + 40587.0 for y in modern_flares_years])

print(f">>> Loaded {len(obs_mjd)} modern high-precision flares for OJ 287.")

# --- 3. RELATIONAL ALGEBRAIC TIMING (STRICT WILL NOMENCLATURE) ---
def will_oj287_strict(params):
    beta_orb, beta_spin, e, omega_i, T_days, t_0 = params

    # 1. Chiral Asymmetry Superposition
    beta_Sigma = beta_orb + beta_spin

    # 2. Relational Spacetime Phase Factor (squared)
    tau_Y_sq = 3 * beta_Sigma**2 - 2 * beta_Sigma**4

    if tau_Y_sq / (1 - e**2) >= 1.0 or e >= 1.0 or e <= 0 or beta_orb <= 0:
        return np.inf

    k_values = np.arange(-30, 30)

    # 3. Absolute phase (o) at disk crossing nodes
    o = k_values * np.pi - omega_i

    # 4. Precession shift explicitly proportional to absolute phase (o)
    omega_shift = (tau_Y_sq / (1 - e**2)) * o

    # 5. Precessing Phase (O)
    O = o - omega_shift

    # 6. Calculate the Stretched Cycle (O_r) using Anton's exact formula
    o_test = 2 * np.pi
    omega_shift_test = (tau_Y_sq / (1 - e**2)) * o_test
    O_r = (2 * np.pi * o_test) / (o_test - omega_shift_test)

    # --- TIME EVOLUTION MAPPING ---
    orbits = np.floor(O / (2*np.pi))
    O_mod = O % (2*np.pi)

    E_mod = 2 * np.arctan(np.sqrt((1-e)/(1+e)) * np.tan(O_mod/2))
    E_mod = np.where(E_mod < 0, E_mod + 2*np.pi, E_mod)

    M_mod = E_mod - e * np.sin(E_mod)
    M_total = orbits * 2 * np.pi + M_mod

    # 7. Time prediction normalized by the Exact Stretched Cycle (O_r)
    t_pred = t_0 + (M_total / O_r) * T_days

    return t_pred

def objective(params):
    t_pred = will_oj287_strict(params)
    if t_pred is np.inf:
        return 1e10

    chi2 = 0
    ERROR_MARGIN = 5.0 # Tightened margin for modern precision

    for obs_t in obs_mjd:
        closest_diff = np.min(np.abs(t_pred - obs_t))
        chi2 += (closest_diff / ERROR_MARGIN)**2

    return chi2

# --- 4. EXECUTE OPTIMIZATION ---
# Initial Guess based on previous successful lock
p0 = [0.100, 0.010, 0.61, 2.2, 4300.0, 58000.0]
bounds = [(0.09, 0.13), (0.005, 0.1), (0.60, 0.75), (0, 2*np.pi), (4200, 4800), (40000, 60000)]

print(">>> Phase 1: Global Relational Search (Modern Epoch)...")
res_nm = minimize(objective, p0, bounds=bounds, method='Nelder-Mead', options={'maxiter': 25000})

print(">>> Phase 2: Precision Algebraic Lock...")
res = minimize(objective, res_nm.x, bounds=bounds, method='L-BFGS-B')

# --- 5. POST-FIT RELATIONAL DERIVATION ---
beta_orb, beta_spin, e, omega_i, T_days, t_0 = res.x
chi2_val = res.fun
t_pred = will_oj287_strict(res.x)

# System Scale (Rs)
T_sec = T_days * 86400
Rs_km = (T_sec * C_KMS * (beta_orb**3)) / np.pi
a_km = Rs_km / (2 * (beta_orb**2))
Mass_Msun = (Rs_km / 2) / MSUN_GM

beta_Sigma = beta_orb + beta_spin
tau_Y_sq = 3 * beta_Sigma**2 - 2 * beta_Sigma**4
prec_deg_per_orbit = (tau_Y_sq / (1 - e**2)) * 360.0

# --- 6. SCIENTIFIC OUTPUT ---
print("\n" + "="*75)
print("WILL RELATIONAL GEOMETRY: MODERN EPOCH SOLUTION (10 POINTS)")
print("="*75)
print(f"Fit Quality (Chi-Squared):       {chi2_val:.4f}")
print("-" * 75)
print(f"Orbital Projection (beta_orb):   {beta_orb:.6f}")
print(f"Spin Projection (beta_spin):     {beta_spin:.6f}")
print(f"Composite Projection (beta_Sig): {beta_Sigma:.6f}")
print(f"Eccentricity (e):                {e:.6f}")
print(f"Initial Phase Turn (omega_i):    {omega_i:.6f} rad")
print(f"Structural Period (T):           {T_days/365.25:.4f} Years")
print(f"Precession Shift per Orbit:      {prec_deg_per_orbit:.2f} Degrees")
print("-" * 75)
print("DERIVED PHYSICAL PARAMETERS (FROM T and beta_orb ONLY):")
print(f"Schwarzschild Radius (Rs):       {Rs_km:.2e} km")
print(f"Total System Mass (M1 + M2):     {Mass_Msun/1e9:.4f} Billion M_sun")
print("="*75)

# --- 7. TIMING ACCURACY ---
matched_preds = []
for obs_t in obs_mjd:
    idx = np.argmin(np.abs(t_pred - obs_t))
    matched_preds.append(t_pred[idx])

df_results = pd.DataFrame({
    'Observed Year': [1970 + (t - 40587)/365.25 for t in obs_mjd],
    'WILL Prediction': [1970 + (t - 40587)/365.25 for t in matched_preds],
    'Delta (Days)': np.abs(obs_mjd - matched_preds)
})
print("\n>>> TIMING ACCURACY:")
print(df_results.to_string(index=False, float_format="%.2f"))

# --- 8. FORWARD PREDICTIONS (NEXT TWO FLARES) ---
# Extract the predictions that occur strictly after the last known observation
last_observation_mjd = np.max(obs_mjd)
future_predictions_mjd = np.sort(t_pred[t_pred > last_observation_mjd])

# Select the immediate next two events
next_two_mjd = future_predictions_mjd[:2]

print("\n" + "="*75)
print("WILL RELATIONAL GEOMETRY: FUTURE PREDICTIONS")
print("="*75)
for i, mjd_val in enumerate(next_two_mjd):
    year_val = 1970.0 + (mjd_val - 40587.0) / 365.25
    print(f"Predicted Future Flare {i+1}: Fractional Year {year_val:.4f} (MJD: {mjd_val:.2f})")
print("="*75)

>>> Loaded 10 modern high-precision flares for OJ 287.
>>> Phase 1: Global Relational Search (Modern Epoch)...
>>> Phase 2: Precision Algebraic Lock...

WILL RELATIONAL GEOMETRY: MODERN EPOCH SOLUTION (10 POINTS)
Fit Quality (Chi-Squared):       326.8791
---------------------------------------------------------------------------
Orbital Projection (beta_orb):   0.119348
Spin Projection (beta_spin):     0.005846
Composite Projection (beta_Sig): 0.125194
Eccentricity (e):                0.671192
Initial Phase Turn (omega_i):    3.045869 rad
Structural Period (T):           13.1417 Years
Precession Shift per Orbit:      30.48 Degrees
---------------------------------------------------------------------------
DERIVED PHYSICAL PARAMETERS (FROM T and beta_orb ONLY):
Schwarzschild Radius (Rs):       6.73e+10 km
Total System Mass (M1 + M2):     22.7812 Billion M_sun

>>> TIMING ACCURACY:
 Observed Year  WILL Prediction  Delta (Days)
       1971.13          1971.22         32.17
       1972.97 